# A/B Testing: Frequentist vs. Bayesian Approach

## Based on "How to run A/B Tests as a Data Scientist!" Tutorial

### Link: https://www.youtube.com/watch?v=OVgi6ftJiyQ

In [1]:
import pandas as pd
import numpy as np
import datetime
from scipy.stats import chi2_contingency, beta
from IPython.display import Image

In [ ]:
pwd

In [2]:
cd '/Users/aaronwoodward/Downloads'

/Users/aaronwoodward/Downloads


In [3]:
#Data Collection

df = pd.read_csv('https://raw.githubusercontent.com/ajhalthor/bayesian-testing/main/ab_data.csv')

In [4]:
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [ ]:
df.sample(2)

In [5]:
start_time = datetime.datetime.strptime(df['timestamp'].min(), '%Y-%m-%d %H:%M:%S.%f')
end_time = datetime.datetime.strptime(df['timestamp'].max(), '%Y-%m-%d %H:%M:%S.%f')
data_duration = (end_time - start_time).days

print(f"Number of unique users in experiment: {df['user_id'].nunique()}")
print(f"Data collected for {data_duration} days")
print(f"Landing pages to compare: {df['landing_page'].unique().tolist()}")
print(f"Percentage of users in control: {round(df[df['group']=='control'].shape[0] * 100 / df.shape[0])}%")


Number of unique users in experiment: 290584
Data collected for 21 days
Landing pages to compare: ['old_page', 'new_page']
Percentage of users in control: 50%


## Data Processing

Dealing with repeated exposures for some users

In [6]:
dup_user_ids = df[df.duplicated('user_id')]

In [7]:
dup_user_ids

,user_id,timestamp,group,landing_page,converted
2656,698120,2017-01-15 17:13:42.602796,control,old_page,0
2893,773192,2017-01-14 02:55:59.590927,treatment,new_page,0
7500,899953,2017-01-07 03:06:54.068237,control,new_page,0
8036,790934,2017-01-19 08:32:20.329057,treatment,new_page,0
10218,633793,2017-01-17 00:16:00.746561,treatment,old_page,0
...,...,...,...,...,...
294308,905197,2017-01-03 06:56:47.488231,treatment,new_page,0
294309,787083,2017-01-17 00:15:20.950723,control,old_page,0
294328,641570,2017-01-09 21:59:27.695711,control,old_page,0
294331,689637,2017-01-13 11:34:28.339532,control,new_page,0


In [8]:
sample = df[df['user_id'].isin([746755,722274])]
sample

,user_id,timestamp,group,landing_page,converted
29073,746755,2017-01-11 01:28:57.083669,control,new_page,1
105487,722274,2017-01-19 01:46:53.093257,control,old_page,0
262554,722274,2017-01-09 21:21:23.638444,control,new_page,0
286566,746755,2017-01-05 03:40:08.457451,control,old_page,0


We're more concerned with the the conversion when users get their first exposure with new landing page. Therefore, we should only keep the first timestamp for each user. To do this, we must do steps:

1. Get timestamp of first exposure.
2. Remove users with multiple buckets.

In [10]:
# Step 1: Get timestamp of first exposure
first_conv = sample.groupby('user_id')['timestamp'].min().to_frame().reset_index()
sample = sample.merge(first_conv, on=['user_id', 'timestamp'])
sample

,user_id,timestamp,group,landing_page,converted
0,722274,2017-01-09 21:21:23.638444,control,new_page,0
1,746755,2017-01-05 03:40:08.457451,control,old_page,0


In [11]:
counter = df['user_id'].value_counts()
(counter > 1).value_counts()

count
False    286690
True       3894
Name: count, dtype: int64

3,894 (1.34%) user_ids have been exposed to the old AND new page. It should be okay to remove them.



In [12]:
#2. Remove users with multiple buckets
valid_users = pd.DataFrame(counter[counter == 1].index, columns=['user_id'])
df = df.merge(valid_users, on=['user_id'])

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286690 entries, 0 to 286689
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       286690 non-null  int64 
 1   timestamp     286690 non-null  object
 2   group         286690 non-null  object
 3   landing_page  286690 non-null  object
 4   converted     286690 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 10.9+ MB


In [15]:
# Add week column to see the data as you would during experiment
df['week'] = df['timestamp'].apply(lambda x: datetime.datetime.strptime(x, '%Y-%m-%d %H:%M:%S.%f').isocalendar()[1])
#df.sample()

In [16]:
df.head()

,user_id,timestamp,group,landing_page,converted,week
0,851104,2017-01-21 22:11:48.556739,control,old_page,0,3
1,804228,2017-01-12 08:01:45.159739,control,old_page,0,2
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0,2
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0,1
4,864975,2017-01-21 01:52:26.210827,control,old_page,1,3


In [17]:
df.sample()

,user_id,timestamp,group,landing_page,converted,week
78815,806619,2017-01-24 10:49:51.096527,treatment,new_page,0,4


In [18]:
df['week'].value_counts()

week
2    91380
3    91056
1    83745
4    20509
Name: count, dtype: int64

### 4. Experiment: Frequentist Approach

In [24]:
#Get Stats
NUM_WEEKS = 4 # Vary number to get experiment data at weekly points in time
experiment_data = df[df['week'] <= NUM_WEEKS]
control = experiment_data[experiment_data['group']=='control']
treatment = experiment_data[experiment_data['group']=='treatment']

control_conv_perc = round(control['converted'].sum() * 100/ control['converted'].count(), 3)
treatment_conv_perc = round(treatment['converted'].sum() * 100/ treatment['converted'].count(), 3)
lift = round(treatment_conv_perc - control_conv_perc, 3)

print(f"Treatment Conversion Rate: {treatment_conv_perc}%")
print(f"Control Conversion Rate: {control_conv_perc}%")
print(f"Lift = {lift}%")


Treatment Conversion Rate: 11.873%
Control Conversion Rate: 12.017%
Lift = -0.144%


### Chi-Squared Test

Null Hypothesis: Control & Treatment are independent.
Alternative Hypothesis: Control & Treatment are not independent.


Contingency table for Chi-squared test. Info about Chi-squared test: https://en.wikipedia.org/wiki/Chi-squared_test

In [25]:
#Contingency Table
control_converted = control['converted'].sum()
treatment_converted = treatment['converted'].sum()
control_non_converted = control['converted'].count() - control_converted
treatment_non_converted = treatment['converted'].count() - treatment_converted
contingency_table = np.array([[control_converted, control_non_converted],
                             [treatment_converted, treatment_non_converted]])


In [26]:
contingency_table

array([[ 17220, 126073],
       [ 17025, 126372]])

In [27]:
chi, p_value, _, _ = chi2_contingency(contingency_table, correction=False)

In order to reject the null hypothesis, we need to have the p_value < 0.05 significance level.

In [28]:
chi, p_value

(1.426794609399621, 0.23228827305833816)

Since the p_value > 0.05 we fail to reject the null hypothesis. Hence, we cannot conclude whether there is a relationship between the control and treatment groups.

In [29]:
print(f"{round(p_value * 100, 2)}% probability that a more extreme chi square than {round(chi, 3)} would have occured by chance")

23.23% probability that a more extreme chi square than 1.427 would have occured by chance


But this is tough to interpret. We would to say something about the actual maginitude of lift. Something like this:

In [30]:
print(f"(We CANNOT say this) We are {round(p_value * 100, 2)}% confident that our lift = {lift}%")

(We CANNOT say this) We are 23.23% confident that our lift = -0.144%


### 5. Bayesian Approach

We want to input the prior distribution and have the experiment update the parameters to create posterier distributions. Since these prior & posterior distributions will be used to sample Conversion Rate, we model them after beta distribtion.

Let's create the prior beta distribtion from the first weeks of conversion data.


In [31]:
prior = df[(df['week'] == 1) & (df['group']=='control')]

In [32]:
prior_means = []

for i in range(10000):
    prior_means.append(prior.sample(1000)['converted'].mean())

In [33]:
prior_means[:10]

[0.109, 0.104, 0.117, 0.125, 0.133, 0.114, 0.11, 0.131, 0.114, 0.113]

In [34]:
# Model Beta Distribution from sample means
prior_alpha, prior_beta, _, _ = beta.fit(prior_means, floc=0, fscale=1)

In [35]:
#Get Stats
NUM_WEEKS = 4 # Vary number to get experiment data at weekly points in time
experiment_data = df[(df['week'] > 1) & (df['week'] <= NUM_WEEKS)]
control = experiment_data[experiment_data['group']=='control']
treatment = experiment_data[experiment_data['group']=='treatment']

control_conv_perc = round(control['converted'].sum() * 100/ control['converted'].count(), 3)
treatment_conv_perc = round(treatment['converted'].sum() * 100/ treatment['converted'].count(), 3)
lift = round(treatment_conv_perc - control_conv_perc, 3)

print(f"Treatment Conversion Rate: {treatment_conv_perc}%")
print(f"Control Conversion Rate: {control_conv_perc}%")
print(f"Lift = {lift}%")

Treatment Conversion Rate: 11.909%
Control Conversion Rate: 12.058%
Lift = -0.149%


In [36]:
control_converted = control['converted'].sum()
treatment_converted = treatment['converted'].sum()
control_non_converted = control['converted'].count() - control_converted
treatment_non_converted = treatment['converted'].count() - treatment_converted

# Update prior parameters with experiment conversion rates
posterior_control = beta(prior_alpha + control_converted, prior_beta + control_non_converted)
posterior_treatment = beta(prior_alpha + treatment_converted, prior_beta + treatment_non_converted)

#Sample from posteriors
control_samples = posterior_control.rvs(1000)
treatment_samples = posterior_treatment.rvs(1000)
probability = np.mean(treatment_samples > control_samples)
print(f"Probability that treatment > control: {probability * 100}%")

Probability that treatment > control: 15.7%


In [37]:
(control_mu), (control_var) = posterior_control.stats()
(treatment_mu), (treatment_var) = posterior_treatment.stats()
print(f"Control Posterior: Mean: {control_mu}, Variance: {control_var}") 
print(f"Treatment Posterior: Mean: {treatment_mu}, Variance: {treatment_var}") 

Control Posterior: Mean: 0.12056205857381884, Variance: 1.0334746117274042e-06
Treatment Posterior: Mean: 0.11909329994574383, Variance: 1.0243764959808975e-06


In [38]:
lift_perc = (treatment_samples - control_samples) / control_samples
print(f"Probability that we are seeing a 2% lift: {np.mean((100 * lift_perc) > 2) * 100}%")

Probability that we are seeing a 2% lift: 0.6%


Advantages of Bayesian over Frequentist:

Results are more interpretable than the ones we got from the frequentist approach
We can interpret results at any point during the experiment. Don't need to wait for an arbitrary "statsig"